### Introduction to NVIDIA cuOpt and MIP

NVIDIA cuOpt is a high-performance, GPU-accelerated combinatorial optimization engine. Unlike traditional solvers strictly focused on CPUs (such as Gurobi or CPLEX in their default modes), which expend significant computational time trying to mathematically prove the exact optimal bound through complex Branch and Bound routines, cuOpt's MIP (Mixed-Integer Programming) module focuses its horsepower on hardware-accelerated primal heuristics. It utilizes a powerful combination of evolutionary algorithms, parallel domain propagation, and the Primal-Dual Hybrid Gradient (PDLP) method. This architecture allows it to find extremely high-quality feasible solutions in fractions of a second, even for industrial-scale problems with millions of variables and constraints.

#### Crop Planning (Profit Maximization)


1. The Core Objective

The goal was to determine the optimal crop planning strategy—specifically, how many whole hectares of Soybeans (\(x\)) and Corn (\(y\)) to plant—in order to maximize the total net profit for the harvest season.

2. The Economic Profile

Each crop presented a different financial return and required a different level of upfront investment in fertilizers and inputs:
• Soybeans: Yielded a high profit of R$ 5,000 per hectare, while costing only R$ 200 per hectare in inputs.
• Corn: Yielded a lower profit of R$ 3,000 per hectare, while costing a higher R$ 400 per hectare in inputs.

3. The Operational Constraints (The Bottlenecks)

The farm could not scale production infinitely because it was restricted by two strict resource walls:
1. Land Limit: The total available arable land was strictly capped at 150 hectares.
2. Input Budget: The total working capital available for buying fertilizer and crop inputs was strictly limited to R$ 40,000.

##### Libraries

In [24]:
from cuopt.linear_programming.problem import Problem, INTEGER, MAXIMIZE
from cuopt.linear_programming.solver_settings import SolverSettings

##### Instantiating the Problem

This line initializes the core data structure in Python that acts as a bridge to NVIDIA’s native C++/CUDA engine. cuOpt allocates memory arrays to host variables, objective parameters, and constraint blocks, packaging them for high-speed GPU transfer.

In [25]:
prob_plantio = Problem("Planejamento_Plantio_Agro")


##### Declaring Variables

In [ ]:
# 1. Defina explicitamente os limites (mínimo 0, máximo None/infinito)
x_soja = prob_plantio.addVariable(vtype=INTEGER, lb=0, ub=100000, name="Hectares_Soja")
y_milho = prob_plantio.addVariable(vtype=INTEGER, lb=0, ub=100000, name="Hectares_Milho")

• vtype=INTEGER: Instructs the engine that fractional hectares are invalid. The solution must yield discrete, whole integers.

• lb=0: The lower bound. It prevents the mathematical engine from proposing physically impossible negative field areas.

• ub=100000: The upper bound. This parameter solved our previous bottleneck. By substituting the default null value with a high numeric threshold, you unlocked the variable, allowing the primal heuristics to scale and freely search the decision space.

##### Objective Function

• This sets up your algebraic Objective Function. cuOpt compiles this expression into a dense coefficient array. The MAXIMIZE parameter instructs the GPU cores to evaluate combinations that push the scalar product of this matrix to its mathematical ceiling.

In [26]:
# Função Objetivo (Lucros: Soja R$5000/ha, Milho R$3000/ha)
prob_plantio.setObjective(5000 * x_soja + 3000 * y_milho, sense=MAXIMIZE)

##### Constraints

• These boundary conditions define your feasible region. cuOpt converts these inequalities into an optimized sparse constraint matrix. Any crop combination that breaches either the 150-hectare surface limit or the R$ 40,000 cash budget is discarded instantly by the domain propagation algorithms.

In [27]:
# 3. Adicione as restrições explicitamente
prob_plantio.addConstraint(x_soja + y_milho <= 150, name="Limite_Terra")
prob_plantio.addConstraint(200 * x_soja + 400 * y_milho <= 40000, name="Orcamento_Insumos")


##### Solving

This acts as the processing trigger. Python yields control to the hardware. The Papilo presolver simplifies structural relationships on the CPU, and then hands the compressed matrix over to the CUDA Cores and Tensor Cores of your NVIDIA A10 to run massive genetic and primal-dual algorithms in milliseconds.

In [28]:
settings = SolverSettings()
#settings.time_limit = 10.0
solucao = prob_plantio.solve(settings)

cuOpt version: 26.8.0, git hash: 400863c1, host arch: x86_64, device archs: 70-real,75-real,80-real,86-real,90a-real,100f-real,120a-real,120
CPU: Intel(R) Xeon(R) Platinum 8358 CPU @ 2.60GHz, threads (physical/logical): 15/30, RAM (available/total): 228.50 / 235.93 GiB
CUDA 12.9, device: NVIDIA A10 (ID 0), VRAM: 22.06 GiB
CUDA device UUID: 640e1dfe-7d2a-a96b-d7b5-da88e8a1a4a9
Solving a problem with 2 constraints, 2 variables (2 integers), and 4 nonzeros
Problem scaling:
Objective coefficents range:          [3e+03, 5e+03]
Constraint matrix coefficients range: [1e+00, 4e+02]
Constraint rhs / bounds range:        [0e+00, 4e+04]
Variable bounds range:                [0e+00, 1e+05]

MIP row scaling completed
New solution from early primal heuristics (CPUFJ). Objective +7.500000e+05. Time 0.00

Running Papilo presolve (git hash 32b3a87d)
Presolve status: reduced the problem
Presolve removed: 2 constraints, 2 variables, 4 nonzeros
Presolved problem: 0 constraints, 0 variables (0 integer), 0 

• What it means: The cuopt backend successfully linked up with your system architecture.

• The Details: It detected you are using cuOpt version 26.8.0 compiled with CUDA 12.9. It mapped out your system's massive 235 GiB of RAM and assigned the heavy mathematical calculations to your enterprise-grade NVIDIA A10 GPU, which has 22.06 GiB of dedicated ultra-fast video memory (VRAM).


	• 2 constraints, 2 variables: It recognized your two land/budget rules and your two crop choices (Soy and Corn).
	• Objective coefficients range: [3e+03, 5e+03]: Success! This shows scientific notation for \([3000, 5000]\). The GPU correctly loaded your profit metrics.
	• Variable bounds range: [0e+00, 1e+05]: Success! This reflects your lb=0 and ub=100000 changes. The variables are now completely untrapped and free to scale.
	• MIP row scaling completed: The engine scaled down the wide gap between small numbers (like 1) and big numbers (like 40,000) so that floating-point math calculations don't suffer from rounding errors on the GPU.


In [29]:
if prob_plantio.Status.name in ["Optimal", "FeasibleFound"]:
    print("--- SOLUÇÃO ENCONTRADA COM SUCESSO ---")
    print(f"Status da Solução: {prob_plantio.Status.name}")
    print(f"Hectares de Soja a plantar: {x_soja.getValue()}")
    print(f"Hectares de Milho a plantar: {y_milho.getValue()}")
    
    # Formatação numérica isolada para evitar erros de caractere de escape na f-string
    lucro_formatado = "{:,.2f}".format(prob_plantio.ObjValue)
    print(f"Lucro Máximo Projetado: R$ {lucro_formatado}")
else:
     print(f"O solucionador não encontrou uma solução viável. Status: {prob_plantio.Status.name}")

--- SOLUÇÃO ENCONTRADA COM SUCESSO ---
Status da Solução: Optimal
Hectares de Soja a plantar: 150.0
Hectares de Milho a plantar: 0.0
Lucro Máximo Projetado: R$ 750,000.00
